# AEGEAN | eSky Scraper

Collect ratings and optional written reviews using Selenium pagination.

**Dependencies:** selenium beautifulsoup4 pandas. Install in the active kernel before running.

**Execution:** Run the setup, configuration and collection cells in order. Review airline selections and limits before collection. Existing data must be migrated to `Data/Scraping/esky/` before resuming.

**Outputs:** All generated files are stored in `Data/Scraping/esky/`. Existing filenames and schemas are retained.

Collection requires network access; browser-based scrapers also require Chrome.

## 1. Portable project paths

In [ ]:
from pathlib import Path

# Locate the project from the notebook kernel's working directory.
# Start Jupyter inside the project tree; the root name is not significant.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if all((p / name).is_dir() for name in ("Python Code", "Data", "Streamlit"))),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the project folder containing Python Code, Data and Streamlit.")

SCRAPING_DIR = PROJECT_ROOT / "Data" / "Scraping"
SOURCE_DIR = SCRAPING_DIR / 'esky'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)


## Imports and prerequisites

In [ ]:
# ============================================================
# eSky AIRLINE REVIEWS SCRAPER - FINAL VERSION
# ============================================================
#
# Features:
# 1. Multiple airline URLs
# 2. All available pages (configurable)
# 3. Reviews with AND without written comments
# 4. Overall rating and eight rating categories
# 5. Safe pagination - avoids skipped pages
# 6. Automatic retries
# 7. Deduplication using original eSky review IDs
# 8. Incremental CSV saving
# 9. Preserves existing reviews
# 10. Diagnostic HTML when errors occur
# 11. No publication-date filtering
# 12. NLP-ready has_review_text indicator
#
# Install dependencies if needed:
# %pip install selenium beautifulsoup4 pandas
#
# ============================================================

import re
import time
import random
import hashlib

from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlsplit, urlunsplit

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    StaleElementReferenceException,
    WebDriverException,
)

## 1. Configuration

In [ ]:
URLS = [
    "https://www.esky.eu/reviews/al/a3/aegean",
    "https://www.esky.eu/reviews/al/tk/turkish-airlines",
    "https://www.esky.eu/reviews/al/af/air-france",
    "https://www.esky.eu/reviews/al/lh/lufthansa",
    "https://www.esky.eu/reviews/al/pg/bangkok-airways",
    "https://www.esky.eu/reviews/al/pd/porter-airlines",
    "https://www.esky.eu/reviews/al/va/virgin-australia",
]

# None = scrape all available pages.
# Example: 22 = scrape first 22 pages.
MAX_PAGES = None

OUTPUT_FILE = SOURCE_DIR / "esky_all_reviews_v2.csv"

DEBUG_DIR = SOURCE_DIR / "esky_debug"
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

HEADLESS = False

PAGE_TIMEOUT = 30
INITIAL_WAIT = 3

MIN_DELAY = 2
MAX_DELAY = 4

MAX_RETRIES = 3

# Preserve existing CSV records.
RESUME = True

## 2. Output Columns

In [ ]:
COLUMNS = [
    "review_id",
    "source_review_id",
    "airline",
    "author",
    "country_code",
    "publication_date",
    "publication_date_text",
    "review_title",
    "overall_rating",
    "rating_overall_category",
    "rating_staff",
    "rating_punctuality",
    "rating_flights_network",
    "rating_ticket_prices",
    "rating_travel_comfort",
    "rating_luggage_carriage",
    "rating_meals",
    "helpful_votes",
    "review_text",
    "has_review_text",
    "source",
    "source_url",
]

## 3. Browser

In [ ]:
def create_driver():

    """Configure and initialize the Chrome WebDriver."""
    options = Options()

    options.add_argument("--start-maximized")
    options.add_argument("--disable-notifications")
    options.add_argument("--lang=en-GB")

    if HEADLESS:
        options.add_argument("--headless=new")

    driver = webdriver.Chrome(options=options)

    driver.set_page_load_timeout(60)

    return driver

## 4. Helpers

In [ ]:
def clean_text(value):

    """Normalize whitespace in a possibly missing text value."""
    return re.sub(
        r"\s+",
        " ",
        value or ""
    ).strip()


def normalize_url(url):

    """Canonicalize the URL for consistent review identification."""
    parts = urlsplit(url.strip())

    return urlunsplit((
        parts.scheme,
        parts.netloc.lower(),
        parts.path.rstrip("/"),
        parts.query,
        ""
    ))


def extract_airline(url):

    """Derive the airline label from its URL slug."""
    slug = (
        urlsplit(url)
        .path.rstrip("/")
        .split("/")[-1]
    )

    return slug.replace("-", " ").title()


def parse_number(value):

    """Convert a rating to a numeric value when possible."""
    if value is None:
        return None

    try:
        return float(
            clean_text(str(value)).replace(",", ".")
        )

    except (ValueError, TypeError):
        return None


def get_text(card, selector):

    """Extract text from the first matching HTML element."""
    node = card.select_one(selector)

    if node:
        return clean_text(
            node.get_text(" ", strip=True)
        )

    return None


def make_review_id(
    url,
    source_review_id,
    author,
    publication_date,
    review_text
):

    """Construct a stable identifier for deduplication."""
    if source_review_id:

        raw = (
            f"{normalize_url(url)}|"
            f"{source_review_id}"
        )

    else:

        raw = (
            f"{normalize_url(url)}|"
            f"{author}|"
            f"{publication_date}|"
            f"{review_text}"
        )

    return hashlib.sha256(
        raw.encode("utf-8")
    ).hexdigest()[:20]

## 5. Review Extraction

In [ ]:
RATING_MAPPING = {
    "overall": "rating_overall_category",
    "staff": "rating_staff",
    "punctuality": "rating_punctuality",
    "flights network": "rating_flights_network",
    "ticket prices": "rating_ticket_prices",
    "travel comfort": "rating_travel_comfort",
    "luggage carriage": "rating_luggage_carriage",
    "meals": "rating_meals",
}


def extract_reviews(html, url):

    """Parse review cards into structured records."""
    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    airline = extract_airline(url)

    cards = soup.select("li.review")

    results = []
    seen_ids = set()

    for card in cards:

        # ----------------------------------------
        # Author
        # ----------------------------------------

        author = get_text(
            card,
            ".info-user .user-name"
        )

        # ----------------------------------------
        # Publication date
        # ----------------------------------------

        date_node = card.select_one(
            ".published-date"
        )

        publication_date = None
        publication_date_text = None

        if date_node:

            publication_date_text = clean_text(
                date_node.get_text()
            )

            timestamp = date_node.get(
                "data-publish-date"
            )

            if timestamp:

                try:

                    publication_date = (
                        datetime.fromtimestamp(
                            int(timestamp),
                            tz=timezone.utc
                        ).strftime("%Y-%m-%d")
                    )

                except (
                    ValueError,
                    OverflowError,
                    OSError
                ):
                    pass

        # ----------------------------------------
        # Country
        # ----------------------------------------

        country_node = card.select_one(
            ".country-date .flag-icon"
        )

        country_code = (
            country_node.get("title")
            if country_node else None
        )

        # ----------------------------------------
        # Review title
        # ----------------------------------------

        review_title = get_text(
            card,
            ".review-title"
        )

        # ----------------------------------------
        # Overall rating
        # ----------------------------------------

        overall_rating = parse_number(
            get_text(
                card,
                ".review-head > .rating"
            )
        )

        # ----------------------------------------
        # Rating categories
        # ----------------------------------------

        subratings = {
            col: None
            for col in RATING_MAPPING.values()
        }

        for item in card.select(
            ".rating-details li.rating-item"
        ):

            name = get_text(
                item,
                ".rating-text .name"
            )

            value = get_text(
                item,
                ".rating-text .value"
            )

            if not name:
                continue

            name = (
                name.lower()
                .rstrip(":")
                .strip()
            )

            if name in RATING_MAPPING:

                subratings[
                    RATING_MAPPING[name]
                ] = parse_number(value)

        # ----------------------------------------
        # Review text
        #
        # IMPORTANT FIX:
        # Keep reviews even when text is empty.
        # ----------------------------------------

        review_text = get_text(
            card,
            ".review-content"
        ) or ""

        has_review_text = bool(
            review_text.strip()
        )

        # ----------------------------------------
        # Original eSky review ID
        # ----------------------------------------

        button = card.select_one(
            ".helpful-button[data-review-id]"
        )

        source_review_id = (
            button.get("data-review-id")
            if button else None
        )

        # ----------------------------------------
        # Helpful votes
        # ----------------------------------------

        helpful_votes = None

        votes_text = get_text(
            card,
            ".helpful-votes"
        )

        if votes_text:

            try:
                helpful_votes = int(
                    votes_text.replace(",", "")
                )
            except ValueError:
                pass

        # ----------------------------------------
        # Unique ID
        # ----------------------------------------

        review_id = make_review_id(
            url,
            source_review_id,
            author,
            publication_date,
            review_text
        )

        if review_id in seen_ids:
            continue

        seen_ids.add(review_id)

        # ----------------------------------------
        # Store review
        # ----------------------------------------

        results.append({

            "review_id": review_id,

            "source_review_id": source_review_id,

            "airline": airline,

            "author": author,

            "country_code": country_code,

            "publication_date": publication_date,

            "publication_date_text": publication_date_text,

            "review_title": review_title,

            "overall_rating": overall_rating,

            **subratings,

            "helpful_votes": helpful_votes,

            "review_text": review_text,

            "has_review_text": has_review_text,

            "source": "eSky",

            "source_url": normalize_url(url),

        })

    return results

## 6. Pagination Helpers

In [ ]:
def get_current_page(driver):

    """Read the active pagination index."""
    try:

        element = driver.find_element(
            By.CSS_SELECTOR,
            "div.pagination li.selected"
        )

        return int(
            element.text.strip()
        )

    except (
        NoSuchElementException,
        StaleElementReferenceException,
        ValueError
    ):
        return None


def get_total_pages(driver):

    """Determine the available pagination range."""
    try:

        element = driver.find_element(
            By.CSS_SELECTOR,
            "ul.qa-number-of-all-pages"
        )

        return int(
            element.get_attribute(
                "data-qa-number-of-all-pages"
            )
        )

    except (
        NoSuchElementException,
        StaleElementReferenceException,
        ValueError,
        TypeError
    ):
        return 1


def get_review_ids(driver, url):

    """Extract identifiers from the currently loaded reviews."""
    reviews = extract_reviews(
        driver.page_source,
        url
    )

    return {
        r["review_id"]
        for r in reviews
    }


def wait_for_reviews(
    driver,
    url,
    timeout=30
):

    """Wait for review content to become available or change."""
    WebDriverWait(
        driver,
        timeout,
        poll_frequency=1
    ).until(
        lambda d: len(
            extract_reviews(
                d.page_source,
                url
            )
        ) > 0
    )

## 7. Safe Pagination

In [ ]:
def go_to_next_page(driver, url):

    """Advance pagination and verify the new page."""
    current_page = get_current_page(
        driver
    )

    if current_page is None:

        print(
            "Cannot determine current page."
        )

        return False

    target_page = current_page + 1

    old_ids = get_review_ids(
        driver,
        url
    )

    print(
        f"Navigating from page "
        f"{current_page} to {target_page}"
    )

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        print(
            f"Attempt {attempt}/{MAX_RETRIES}"
        )

        actual_page = get_current_page(
            driver
        )

        # ----------------------------------------
        # Click only if still on original page
        # ----------------------------------------

        if actual_page == current_page:

            try:

                next_button = driver.find_element(
                    By.CSS_SELECTOR,
                    "div.pagination a.next"
                )

                driver.execute_script(
                    "arguments[0].scrollIntoView({block:'center'});",
                    next_button
                )

                time.sleep(0.5)

                driver.execute_script(
                    "arguments[0].click();",
                    next_button
                )

            except (
                StaleElementReferenceException,
                WebDriverException
            ) as exc:

                print(
                    "Click issue:",
                    type(exc).__name__
                )

        elif actual_page == target_page:

            print(
                f"Already on page {target_page}; "
                "not clicking again."
            )

        else:

            print(
                f"Unexpected page position: "
                f"{actual_page}"
            )

            return False

        # ----------------------------------------
        # Wait for correct selected page
        # ----------------------------------------

        try:

            WebDriverWait(
                driver,
                PAGE_TIMEOUT,
                poll_frequency=1
            ).until(
                lambda d:
                get_current_page(d) == target_page
            )

        except TimeoutException:

            print(
                f"Page {target_page} "
                "did not become selected."
            )

            time.sleep(2)

            continue

        # ----------------------------------------
        # Wait for reviews
        # ----------------------------------------

        try:

            WebDriverWait(
                driver,
                PAGE_TIMEOUT,
                poll_frequency=1
            ).until(
                lambda d:
                len(
                    extract_reviews(
                        d.page_source,
                        url
                    )
                ) > 0
            )

        except TimeoutException:

            print(
                f"Page {target_page} selected, "
                "but reviews did not load."
            )

            time.sleep(3)

            continue

        # ----------------------------------------
        # Validate IDs
        # ----------------------------------------

        new_ids = get_review_ids(
            driver,
            url
        )

        unseen_ids = new_ids - old_ids

        print(
            f"Page {target_page}: "
            f"{len(new_ids)} reviews, "
            f"{len(unseen_ids)} IDs different "
            "from previous page"
        )

        if len(unseen_ids) == 0:

            print(
                "WARNING: Review IDs match "
                "the previous page."
            )

            return False

        print(
            f"Navigated: Page {current_page} "
            f"→ {target_page}"
        )

        return True

    return False

## 8. Diagnostics

In [ ]:
def save_diagnostic_html(
    driver,
    airline,
    page
):

    """Persist browser HTML for troubleshooting."""
    filename = (
        DEBUG_DIR /
        f"{airline.lower().replace(' ', '_')}"
        f"_page_{page}_error.html"
    )

    html = driver.page_source

    filename.write_text(
        html,
        encoding="utf-8"
    )

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    print("\nDIAGNOSTIC INFORMATION")
    print("-" * 50)

    print(
        "li.review elements:",
        len(soup.select("li.review"))
    )

    print(
        "review-content elements:",
        len(soup.select(".review-content"))
    )

    print(
        "review IDs:",
        len(soup.select("[data-review-id]"))
    )

    print(
        "HTML length:",
        len(html)
    )

    print(
        "Selected page:",
        get_current_page(driver)
    )

    print(
        "Current URL:",
        driver.current_url
    )

    print(
        "HTML saved:",
        filename
    )

    return filename

## 9. Save Csv

In [ ]:
def save_reviews(reviews):

    """Merge collected records with the existing CSV and save the result."""
    if not reviews:
        return

    df_new = pd.DataFrame(
        reviews,
        columns=COLUMNS
    )

    path = Path(OUTPUT_FILE)

    if path.exists() and RESUME:

        df_old = pd.read_csv(
            path,
            dtype={
                "review_id": str,
                "source_review_id": str
            }
        )

        for col in COLUMNS:

            if col not in df_old.columns:
                df_old[col] = None

        df = pd.concat(
            [
                df_old[COLUMNS],
                df_new[COLUMNS]
            ],
            ignore_index=True
        )

    else:

        df = df_new

    df.drop_duplicates(
        subset=["review_id"],
        keep="last",
        inplace=True
    )

    df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"CSV updated: "
        f"{len(df)} total unique reviews"
    )

## 10. Scrape Airline

In [ ]:
def scrape_airline(driver, url):

    """Collect reviews for one airline using the configured scraper."""
    url = normalize_url(url)

    airline = extract_airline(url)

    print("\n" + "=" * 65)
    print(f"Scraping: {airline}")
    print(f"URL: {url}")
    print("=" * 65)

    driver.get(url)

    time.sleep(INITIAL_WAIT)

    wait_for_reviews(
        driver,
        url,
        PAGE_TIMEOUT
    )

    total_pages = get_total_pages(
        driver
    )

    pages_to_scrape = (
        min(total_pages, MAX_PAGES)
        if MAX_PAGES is not None
        else total_pages
    )

    print(
        f"Total pages available: "
        f"{total_pages}"
    )

    print(
        f"Pages to scrape: "
        f"{pages_to_scrape}"
    )

    all_reviews = {}

    for page_index in range(
        pages_to_scrape
    ):

        current_page = get_current_page(
            driver
        )

        reviews = extract_reviews(
            driver.page_source,
            url
        )

        # ----------------------------------------
        # Empty page diagnostics
        # ----------------------------------------

        if len(reviews) == 0:

            print(
                f"\nWARNING: No reviews "
                f"extracted on page {current_page}"
            )

            save_diagnostic_html(
                driver,
                airline,
                current_page
            )

            break

        # ----------------------------------------
        # Session deduplication
        # ----------------------------------------

        new_reviews = []

        for review in reviews:

            review_id = review["review_id"]

            if review_id not in all_reviews:

                all_reviews[review_id] = review

                new_reviews.append(review)

        # ----------------------------------------
        # Count reviews with text
        # ----------------------------------------

        text_reviews = sum(
            r["has_review_text"]
            for r in reviews
        )

        rating_only = (
            len(reviews) - text_reviews
        )

        print(
            f"Page {current_page or page_index + 1}"
            f"/{total_pages} | "
            f"Found: {len(reviews)} | "
            f"New: {len(new_reviews)} | "
            f"With text: {text_reviews} | "
            f"Rating only: {rating_only} | "
            f"Session total: {len(all_reviews)}"
        )

        # ----------------------------------------
        # Incremental save
        # ----------------------------------------

        save_reviews(reviews)

        # ----------------------------------------
        # Page limit
        # ----------------------------------------

        if page_index + 1 >= pages_to_scrape:
            break

        # ----------------------------------------
        # Safe pagination
        # ----------------------------------------

        success = go_to_next_page(
            driver,
            url
        )

        if not success:

            print(
                "\nStopping: page transition "
                "could not be verified."
            )

            save_diagnostic_html(
                driver,
                airline,
                get_current_page(driver)
            )

            break

        time.sleep(
            random.uniform(
                MIN_DELAY,
                MAX_DELAY
            )
        )

    print(
        f"\nCompleted {airline}: "
        f"{len(all_reviews)} unique reviews "
        "collected in this session."
    )

    return list(
        all_reviews.values()
    )

## 11. Main

In [ ]:
def main():

    """Orchestrate collection and report the resulting dataset."""
    driver = create_driver()

    total_collected = 0

    try:

        for url in URLS:

            try:

                reviews = scrape_airline(
                    driver,
                    url
                )

                total_collected += len(reviews)

            except Exception as exc:

                print(
                    f"\nError scraping {url}: "
                    f"{exc}"
                )

                try:

                    save_diagnostic_html(
                        driver,
                        extract_airline(url),
                        get_current_page(driver)
                    )

                except Exception:
                    pass

    finally:

        driver.quit()

    print("\n" + "=" * 65)
    print("SCRAPING COMPLETED")
    print("=" * 65)

    print(
        f"Reviews collected this run: "
        f"{total_collected}"
    )

    if Path(OUTPUT_FILE).exists():

        df = pd.read_csv(
            OUTPUT_FILE
        )

        # Recalculate text indicator from the
        # actual text column for consistency.
        df["has_review_text"] = (
            df["review_text"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
        )

        print(
            f"Total unique reviews in CSV: "
            f"{len(df)}"
        )

        print("\nReviews per airline:")

        summary = (
            df.groupby("airline")
            .agg(
                total_reviews=("review_id", "size"),
                reviews_with_text=(
                    "has_review_text", "sum"
                )
            )
        )

        summary["rating_only"] = (
            summary["total_reviews"]
            - summary["reviews_with_text"]
        )

        print(
            summary.sort_values(
                "total_reviews",
                ascending=False
            ).to_string()
        )

        print("\nMissing ratings:")

        rating_columns = [
            col for col in df.columns
            if col == "overall_rating"
            or col.startswith("rating_")
        ]

        print(
            df[rating_columns]
            .isna()
            .sum()
            .to_string()
        )

## 12. Execute

In [ ]:
if __name__ == "__main__":
    main()

## Inspect saved data / checkpoint status

In [ ]:

import pandas as pd

df = pd.read_csv(SOURCE_DIR / "esky_all_reviews_v2.csv")

print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

print("\nReview sample:")
print(
    df[[
        "airline",
        "author",
        "publication_date",
        "review_title",
        "overall_rating",
        "rating_staff",
        "rating_punctuality",
        "rating_travel_comfort",
        "review_text"
    ]].head(10).to_string(index=False)
)

print("\nAverage ratings by airline:")

rating_cols = [
    "overall_rating",
    "rating_staff",
    "rating_punctuality",
    "rating_ticket_prices",
    "rating_travel_comfort",
    "rating_meals"
]

print(
    df.groupby("airline")[rating_cols]
      .mean()
      .round(2)
)

## Inspect saved data / checkpoint status

In [ ]:
df.head()

## Optional diagnostic: historical page 20
Run only when investigating a saved error page. Disabled by default.

In [ ]:
RUN_DIAGNOSTIC = False

if RUN_DIAGNOSTIC:

    from pathlib import Path
    from bs4 import BeautifulSoup
    import pandas as pd

    # Previously saved diagnostic HTML
    html_path = Path(
        SOURCE_DIR / "esky_debug/aegean_page_20_error.html"
    )

    html = html_path.read_text(
        encoding="utf-8"
    )

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    cards = soup.select("li.review")

    print("=" * 70)
    print("PAGE 20 DIAGNOSTIC")
    print("=" * 70)

    print("Total review cards:", len(cards))

    results = []

    for i, card in enumerate(cards, start=1):

        author_node = card.select_one(
            ".info-user .user-name"
        )

        text_node = card.select_one(
            ".review-content"
        )

        rating_node = card.select_one(
            ".review-head > .rating"
        )

        id_node = card.select_one(
            ".helpful-button[data-review-id]"
        )

        author = (
            author_node.get_text(" ", strip=True)
            if author_node else None
        )

        review_text = (
            text_node.get_text(" ", strip=True)
            if text_node else None
        )

        rating = (
            rating_node.get_text(" ", strip=True)
            if rating_node else None
        )

        review_id = (
            id_node.get("data-review-id")
            if id_node else None
        )

        results.append({
            "card": i,
            "source_review_id": review_id,
            "author": author,
            "rating": rating,
            "text_length": len(review_text or ""),
            "review_text": (review_text or "")[:120]
        })

    diagnostic_df = pd.DataFrame(results)

    print(
        diagnostic_df.to_string(index=False)
    )

    print("\n" + "=" * 70)
    print("TESTING EXISTING EXTRACTION FUNCTION")
    print("=" * 70)

    url = "https://www.esky.eu/reviews/al/a3/aegean"

    try:
        extracted = extract_reviews(html, url)

        print("Successfully extracted:", len(extracted))

        if extracted:
            print("\nFirst extracted review:")
            print(extracted[0])

    except Exception as exc:
        print("EXTRACTION ERROR:", repr(exc))